<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" alt="OPIM 5512 banner" width="100%">

# A09 · Forecasting Tomorrow's Peak: Connecticut Electricity from Six Messy Weather Stations
**Individual assignment · Module 4 (Time Series) · 100 points (+5 bonus) · due Friday, November 6, 11:59 PM**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/A09_Peak_Forecast_STARTER.ipynb)

Utilities and grid operators care most about the **peak**: the single highest hour of the day. Peaks decide how many
power plants must be ready, and they drive a big part of everyone's electricity bill. Your job: every evening, forecast
**tomorrow's peak load** for Connecticut, in MW.

Lab 5 handed you clean data from one airport. Real forecasting starts messier: ISO New England combines several weather
stations into one Connecticut temperature, and real sensors break. You get **raw reports from six Connecticut airports**,
three years of them, and the hourly Connecticut load. **We planted five realistic sensor problems** in the weather data
(the kind that really happen). Find them, fix them, and defend your choices.

| Part | Points | What you do |
|---|---|---|
| **1. EDA** | 10 | Look at the raw station data and at three years of daily peaks |
| **2. Clean** | 25 | Find and fix the five planted problems, one report per hour, combine six stations into one Connecticut temperature |
| **3. Features** | 15 | The window method (lags, rolling windows) + TSFresh features from each day's 24 hours |
| **4. Walk-forward validation** | 25 | Baselines, a model trained once vs a model retrained every month, and: did six stations beat one? |
| **5. Interpret** | 15 | Permutation importance (top 10, boxplot) and partial dependence for the top 3 features |
| **6. Conclusions** | 10 | Five bullets: what you learned |
| **Bonus** | +5 | Peak alerts: could your model warn about the hottest days? |

**Rules of the road**
- Runs on **free Colab** in about 5 to 10 minutes. No Google Cloud, no Kaggle, no downloads to your computer.
- Write your own code in the `YOUR CODE HERE` cells (Claude and Gemini are great tutors: ask them to explain, then type it
  yourself). Keep everything that's given.
- **No leakage across time.** A forecast made on the evening of day *d* may use anything up to the end of day *d*, plus the
  **weather forecast** for tomorrow. (We use tomorrow's actual weather as a stand-in for its forecast, like Lab 5.)
- Submit your **fully executed** `.ipynb` on HuskyCT (Module 4, Week 4.2, Submit A09).

# Setup (run me)

In [ ]:
!pip install -q tsfresh

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module4/data/"
STATIONS = {"BDL": "Bradley (Windsor Locks)", "HFD": "Hartford-Brainard", "BDR": "Bridgeport (Sikorsky)",
            "HVN": "New Haven (Tweed)", "GON": "Groton-New London", "DXR": "Danbury"}
SEED = 5512
print("Ready.")

# Part 1 · EDA (10 points)

## 1a. The raw weather
One file, six stations, every report (routine reports at :51 past the hour, plus "special" reports when the weather
changes fast), in UTC. Columns: `tmpf` temperature (°F), `dwpf` dew point (°F), `relh` relative humidity (%), `sknt`
wind (knots), `p01i` rain in the last hour (inches, `T` = trace). `M` means missing. Everything is read as **text** on
purpose, so nothing gets silently converted before you look.

In [ ]:
raw = pd.read_csv(DATA + "a09/ct_airports_asos_2023_2025.csv", dtype=str, keep_default_na=False)
print(len(raw), "reports")
raw.head()

**Your turn (1a):** how many reports does each station have, what is the first and last report time, and what does
one station's temperature look like over one week? Pick any station and any week.

In [ ]:
raw["valid_utc"] = pd.to_datetime(raw["valid"], utc=True)
# YOUR CODE HERE: reports per station, first and last report time per station (hint: groupby + agg)

# YOUR CODE HERE: plot one station's temperature for one week (hint: pd.to_numeric(..., errors="coerce"))


## 1b. The load and the target
Connecticut hourly load from ISO New England (the same file as Lab 5). `begin_date` is local time **with its UTC offset**,
so daylight saving is unambiguous. We convert to UTC, then give every hour its **Connecticut day**: the peak is the highest
hour of each local calendar day.

In [ ]:
load_raw = pd.read_csv(DATA + "raw/ct_hourly_load_2023_2025_raw.csv")
load = pd.DataFrame({"hour_utc": pd.to_datetime(load_raw["begin_date"], utc=True), "load_mw": load_raw["load_mw"]})
load = load.drop_duplicates("hour_utc").set_index("hour_utc").sort_index()
load["local_day"] = load.index.tz_convert("America/New_York").tz_localize(None).normalize()
daily_peak = load.groupby("local_day")["load_mw"].max()
print(len(load), "hours ->", len(daily_peak), "days")

**Your turn (1b):** plot the daily peak for all three years, and print the 5 highest peak days. What do they have
in common (look at the dates)?

In [ ]:
# YOUR CODE HERE: plot daily_peak, then print the 5 highest days


*Your answer (1b, one or two sentences):*

# Part 2 · Clean (25 points)

Step by step. Each step is short; the thinking is the hard part. Write down what you find.

## 2a. Text to numbers
`M` becomes missing, `T` (trace of rain) becomes a tiny amount (0.001 in), everything else becomes a number.

In [ ]:
NUMBER_COLUMNS = ["tmpf", "dwpf", "relh", "sknt", "p01i"]
wx = raw.copy()
wx["p01i"] = wx["p01i"].replace("T", "0.001")
for column in NUMBER_COLUMNS:
    wx[column] = pd.to_numeric(wx[column].replace("M", ""), errors="coerce")
wx[NUMBER_COLUMNS].describe().round(1)

## 2b. Planted problems: find all five
Real sensors break in a handful of typical ways. Look for each one below. For each problem you find, print **the station,
the dates, how many reports**, and fix it. Hints, one per problem type:
1. **Exact duplicate reports** (`duplicated()`).
2. **Impossible values** (no Connecticut airport has reached 120 °F or −40 °F).
3. **A stuck sensor**: the temperature repeats the exact same value for a day or more while the dew point keeps changing
   (on the hourly grid, real temperatures never sit still that long: the longest natural run here is about 16 hours).
4. **Wrong units**: for several days one station reads far colder than all the others (compare each station with the
   median of the six, hour by hour). What unit would make those numbers sensible?
5. **An outage**: a station with no reports at all for days.

Problems 1, 2 and 4 are easiest to find in the raw reports (2b). Problems 3 and 5 are easiest once every station is on an
hourly grid with one report per hour (2c): special reports break up a stuck run, and a missing day only shows up on a grid.

Fix what can be fixed honestly (a unit mistake can be converted back), and set to missing what can't (a stuck reading
carries no information). Missing values get filled later, and with six stations, one missing station is no disaster.

In [ ]:
# Problem 1: exact duplicates (YOUR CODE HERE: count them, print station and dates, drop them)

# Problem 2: impossible values (YOUR CODE HERE: find temperatures above 120 or below -40, print them, set them to missing)



In [ ]:
# Problem 4: wrong units (YOUR CODE HERE)
# Hint: round each report to the nearest hour, average per (hour, station), unstack so the six stations are columns,
#       subtract the row median, and look for long stretches far below it. Then fix those reports.


*Your notes (2b): what did you find for each of the five problems, and why did you fix each one the way you did?*

## 2c. One report per hour per station, on a full grid
The same rule as Lab 5: round each report to the nearest hour (10:51 → 11:00), keep the routine :51 report when there is
one (otherwise the report closest to the hour), then line every station up on the full list of hours. Hours with no report
become missing. Then look for the stuck sensor (problem 3) and the outage (problem 5).

In [ ]:
# YOUR CODE HERE: one report per (station, hour), then pivot temperature and dew point into
#   temp: one column per station, one row per UTC hour on the full grid (use the START and END below)
#   dew:  the same for dew point
START = pd.Timestamp("2023-01-01 00:00", tz="America/New_York").tz_convert("UTC") - pd.Timedelta(hours=72)
END = pd.Timestamp("2025-12-31 23:00", tz="America/New_York").tz_convert("UTC") + pd.Timedelta(hours=72)
grid = pd.date_range(START, END, freq="h")

# YOUR CODE HERE (problem 3): for each station, find runs where the temperature is identical 24+ hours in a row; set them to missing.
# Hint: a new run starts whenever the value changes from the hour before: run_id = (values != values.shift()).cumsum()

# YOUR CODE HERE (problem 5): for each station, find its longest run of missing hours. Which station had an outage, and when?


## 2d. Fill short gaps, then combine six stations into one Connecticut temperature
Fill each station's short gaps (6 hours at most) with time interpolation. Then the **Connecticut temperature** is the
average of the stations that reported that hour, so a long outage at one station simply means five stations that week.
Do the same for dew point. Finally: how different is the Connecticut average from Bradley alone?

In [ ]:
# YOUR CODE HERE: interpolate short gaps (limit=6), then build a table "weather" (index = grid) with columns
#   ct_temp_f, ct_dewpoint_f        (average across stations)
#   bdl_temp_f, bdl_dewpoint_f      (Bradley alone, for a comparison in Part 4)
# and check: no missing values left. How different is Bradley from the six-station average?


# Part 3 · Features (15 points)

## 3a. One row per day
The forecast is made on the evening of day *d*, for day *d+1*. First, a daily table (Connecticut days): the peak, and the
daily weather (high, low, average, highest dew point) from the Connecticut average and from Bradley alone.

In [ ]:
hourly_all = load[["load_mw", "local_day"]].join(weather, how="left")
daily = hourly_all.groupby("local_day").agg(
    peak_mw=("load_mw", "max"),
    ct_tmax=("ct_temp_f", "max"), ct_tmin=("ct_temp_f", "min"), ct_tmean=("ct_temp_f", "mean"),
    ct_dewmax=("ct_dewpoint_f", "max"),
    bdl_tmax=("bdl_temp_f", "max"), bdl_tmin=("bdl_temp_f", "min"), bdl_tmean=("bdl_temp_f", "mean"),
    bdl_dewmax=("bdl_dewpoint_f", "max"))
print(len(daily), "days")
daily.head()

## 3b. The window method
Every feature must be known on the evening of day *d* (or come from tomorrow's weather forecast). Build, for target day
*d+1*:
- **Tomorrow's calendar:** day of week, month, weekend, federal holiday.
- **Tomorrow's weather "forecast":** high, low, average, highest dew point (Connecticut average).
- **Heat buildup:** the average high of the last two days and tomorrow (heat waves get worse as buildings heat up).
- **Lags of the target:** today's peak (`peak_lag1`), and the peak on the same weekday last week (`peak_lag7`, which is
  6 days before today), plus the average peak of the last 7 days.

Use `shift()`: `daily["peak_mw"].shift(1)` lines yesterday's value up with today's row. Name the target column `target`.

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar
holidays = USFederalHolidayCalendar().holidays(start="2022-01-01", end="2026-12-31")

# YOUR CODE HERE: build "rows" (index = the day being forecast) with the column "target" (that day's peak)
# and the window features listed above. Keep their names in a list called WINDOW_FEATURES. Drop rows with missing values.


## 3c. TSFresh: features from each day's 24 hours
Module 4.2's recipe: a long table with an **id** for each sequence and a **time** column to sort by. Here each **day** is
one sequence (its 23, 24 or 25 hours of load and Connecticut temperature). TSFresh summarizes each day's shape (mean,
spread, maximum, ...). Because a forecast for *d+1* may only use day *d*, **shift the TSFresh features by one day** before
joining them to `rows`. `MinimalFCParameters` keeps it fast (about a minute in Colab).

In [ ]:
from tsfresh import extract_features
from tsfresh.feature_extraction import MinimalFCParameters

# YOUR CODE HERE: a long table with local_day (the id), a time column (hour number within the day), load_mw and ct_temp_f;
# extract_features(..., column_id="local_day", column_sort="time", default_fc_parameters=MinimalFCParameters());
# shift the result by ONE day, give the columns a "yday_" prefix, keep their names in TSFRESH_FEATURES, join to rows.


*Your leakage check (3c, two sentences): why must the TSFresh features be shifted by one day, and what would happen to
your test error if you forgot?*

# Part 4 · Walk-forward validation (25 points)

## 4a. Train once vs walk forward
Lab 5 trained once on 2023-2024 and tested on 2025. Real forecasters **retrain as new data arrives**. Walk-forward
validation copies that: for each month of 2025, train on **every day before that month**, forecast the month, move on.
Compare four forecasts on the 365 days of 2025, month by month (MAE in MW):
1. **Persistence:** tomorrow's peak = today's peak (`peak_lag1`)
2. **Same weekday last week** (`peak_lag7`)
3. **Model trained once** on 2023-2024
4. **Model walked forward**, retrained every month

Use the same model settings for 3 and 4 (the given `make_model`).

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

FEATURES = WINDOW_FEATURES + TSFRESH_FEATURES


def make_model():
    return HistGradientBoostingRegressor(learning_rate=0.05, max_iter=400, max_leaf_nodes=15,
                                         min_samples_leaf=10, l2_regularization=1.0, random_state=SEED)


test = rows[rows.index.year == 2025]
train_once = rows[rows.index.year <= 2024]
print(len(train_once), "training days (2023-2024) |", len(test), "test days (2025) |", len(FEATURES), "features")

In [ ]:
# YOUR CODE HERE:
# 1. once_model = make_model(); fit on train_once; predict all of 2025 -> pred_once
# 2. walk forward: for each month 1..12 of 2025, train a fresh make_model() on every row BEFORE the first day of that month,
#    predict that month's rows -> pred_walk (one Series for the whole year)
# 3. a table monthly_mae: one row per month (+ an "ALL 2025" row), one column per forecast (the four above)


In [ ]:
# YOUR CODE HERE: plot the monthly MAE of all four forecasts (one line each, months on the x axis)


## 4b. Did six stations beat one?
Repeat the **walk-forward** forecast with Bradley's weather (`bdl_tmax`, `bdl_tmin`, `bdl_tmean`, `bdl_dewmax`, and a
Bradley heat buildup) in place of the Connecticut average, everything else the same. Was all that cleaning worth it?

In [ ]:
# YOUR CODE HERE: copy rows, swap in Bradley's daily weather (and a Bradley heat_buildup), walk forward again,
# and print both 2025 MAEs side by side


*Your answers (4a and 4b, three or four sentences): which forecast won, in which months did the model struggle most and
why, did retraining every month help, and did six stations beat one?*

# Part 5 · Interpret (15 points)
Use the **model trained once** (`once_model`) on the 2025 test days.
1. **Permutation importance, top 10 features, as a boxplot** (10 shuffles each).
2. **Partial dependence plots for the top 3 features**, and a short description of what each one shows.

In [ ]:
from sklearn.inspection import permutation_importance
# YOUR CODE HERE: permutation importance on test (n_repeats=10), boxplot of the top 10, and a list TOP3


In [ ]:
from sklearn.inspection import PartialDependenceDisplay
# YOUR CODE HERE: partial dependence plots for TOP3 (three panels)


*Your description (Part 5): what does each of the three partial dependence plots show? Does it make physical sense?*

# Part 6 · Conclusions (10 points)
Five bullets: what you learned about cleaning real sensor data, about features for time series, about validating a
forecast honestly, and about this problem.

*Your five bullets:*
1.
2.
3.
4.
5.

# Bonus · Peak alerts (+5 points)
Grid operators issue alerts before the hottest days. Call a day an **alert day** if its peak is above the 95th percentile of
the 2023-2024 daily peaks. Using your walk-forward forecasts: how many of 2025's alert days did your model flag (forecast
above the same line), how many false alarms did it raise, and how does that compare with persistence? Show a confusion matrix.

In [ ]:
# BONUS (optional): YOUR CODE HERE


---
**Data credits:** weather from the Iowa State Mesonet (mesonet.agron.iastate.edu) archive of NWS/FAA airport reports, with
five sensor problems planted for this assignment; load from ISO New England (iso-ne.com, ISO Express web services).